# How findings are classified: the taxonomy

Every finding in a report carries two classifications. The `category`
is a single label, such as `dependencies` or `formats`. The `facets`
are the finding's properties stated as separate fields: what the
statement ranges over, what kind of statement it is, how it is
supported, whether it carries a condition, what role order plays in
it, and whether it reports a structure or a departure. The category
is not assigned independently — it is derived from the facets by a
fixed table, so the label and the fields cannot disagree.

This chapter explains why both classifications exist, defines each
field on findings from a real report, shows how the derivation
works, shows what filtering by facets adds, and states where the
scheme is weak. Every example comes from one profile of the soybean
disease dataset that ships in the repository's benchmark corpus
(683 rows; 36 columns of symptoms and growing conditions, plus the
disease class), run at the standard budget.

## Problem: one label, several kinds of distinction

Before schema 0.3, the category was the only classification, and
each finding was filed under exactly one of 13 categories. Those 13
labels were not one kind of distinction. They mixed at least three:

- **Scope.** Some categories describe a statement about one column
  (a concentration, a format mix); others describe a statement
  about a relation between columns (a dependency, an association).
- **Data-quality dimensions.** `missingness` is the completeness
  dimension and `duplicates` sits under the uniqueness dimension.
  These are properties a statement can have alongside its kind, not
  kinds of statement themselves.
- **Conditionality.** A conditional functional dependency is a
  functional dependency plus a condition. Filing it under its own
  category, `conditional_rules`, made one property — having a
  condition — outrank the statement itself.

The cost of mixing the three is information loss at the boundary.
A dependency that holds only under a condition reads, by category,
as unrelated to the dependencies; a reader filtering on
`dependencies` never sees it, and nothing in its record states that
it is unconditional or conditioned. Facets repair this by giving
each distinction its own field. The category stays, because
existing readers group and filter by it, but it is now computed
from the fields rather than chosen alongside them.

In [1]:
import polars as pl

import onceover

df = pl.read_csv(
    "../../../benchmarks/corpus/data/gritbot/soybean/data.csv"
)
report = onceover.profile(df, budget="standard")
print("schema_version:", report.data["schema_version"])
print("findings selected:", len(report.findings))

schema_version: 0.3
findings selected: 100


## The fields

Six fields are present on every finding. The seventh,
`quality_dimension`, is present only when the finding is a
data-quality statement. Assignment is a property of the finding's
pattern — each finding type has a stated assignment, tabulated in
[the findings reference](https://github.com/fcela/onceover/blob/main/docs/findings.md) —
refined by the finding's own record in four cases:

- A functional dependency is `exact` when its violation count is
  zero, and `approximate` otherwise.
- A conditional rule is `exact` when it has no violations among the
  rows its condition selects, and `approximate` otherwise.
- An extreme value is `value_conditioned` when it is stated within
  a categorical subset, and `unconditional` otherwise.
- A predictive dependency is `segment_conditioned` in its
  piecewise form, and `unconditional` otherwise.

### Field: `scope`

What the statement ranges over. Values: `column` (one column),
`columns` (a relation or combination of columns), `rows` (the row
set), `sequence` (positions along an ordering), and `frame` (the
table as a whole).

In [2]:
def example(field, value):
    hits = [f for f in report.findings if f["facets"].get(field) == value]
    print(f"{field} = {value}: {len(hits)} findings")
    if hits:
        f = hits[0]
        print(f"  [{f['category']}/{f['type']}] {f['claim']}")

example("scope", "column")
example("scope", "columns")
example("scope", "rows")

scope = column: 18 findings
  [concentration/value_concentration] column mycelium is nearly constant: 99.1% 'absent' vs 6 'present'
scope = columns: 81 findings
  [dependencies/functional] int_discolor → sclerotia (sclerotia is a function of int_discolor; 0 violations in 645 rows)
scope = rows: 1 findings
  [duplicates/duplicate_rows] 52 rows repeat earlier values across the compared columns (41 groups, largest 4)


No finding in this report has `sequence` or `frame` scope. The
soybean table declares no ordering, so no statement ranges over
positions; the `frame` value belongs to findings about the table's
layout, such as the long-format finding in
[Reading findings](findings.ipynb).

### Field: `object`

What kind of statement it is. Values: `distribution` (a statement
about how values are spread), `relation` (a statement about how
columns connect), `structure` (a statement about arrangement — row
repetition, sequence layout), `temporal_behaviour` (a statement
about how a quantity behaves along an ordering), and `role` (a
statement about what a column is).

In [3]:
example("object", "distribution")
example("object", "relation")
example("object", "structure")

object = distribution: 37 findings
  [concentration/value_concentration] column mycelium is nearly constant: 99.1% 'absent' vs 6 'present'
object = relation: 62 findings
  [dependencies/functional] int_discolor → sclerotia (sclerotia is a function of int_discolor; 0 violations in 645 rows)
object = structure: 1 findings
  [duplicates/duplicate_rows] 52 rows repeat earlier values across the compared columns (41 groups, largest 4)


### Field: `evidence`

How the statement is supported. Values: `exact` (it holds without
exception in the evaluated rows), `approximate` (it holds at a
measured rate, or within a tolerance), `model_based` (it rests on a
fitted model), and `inferred` (it is an interpretation drawn from
observations). The first three all occur in this report:

In [4]:
example("evidence", "approximate")
example("evidence", "model_based")

evidence = approximate: 26 findings
  [dependencies/functional] class, leafspots_halo → leafspot_size holds on 99.67% of rows (2 violations in 599 rows)
evidence = model_based: 32 findings
  [dependencies/predictive] int_discolor is predicted from [class] with 1.000 cross-validated accuracy (majority baseline 0.901)


The `inferred` value does not occur here. It belongs to the column
role suggestions — statements such as "this column reads as an
identifier" — and no role finding was selected into this report.

### Field: `conditionality`

Whether the statement carries a condition. Values: `unconditional`
(no condition), `value_conditioned` (the statement holds under a
condition on column values), and `segment_conditioned` (the
statement's behaviour differs across segments of an ordering or of
an input). This field carries the distinction the old scheme
promoted into a category; the pair of findings further down shows
what that costs and what the field recovers.

### Field: `temporality`

The role of order in the statement. Values: `atemporal` (order
plays no role), `ordered` (the statement is about an ordering —
a gap in a sequence, irregular spacing), and
`change_across_segments` (the statement compares segments of an
ordering — a level shift, a drift score). Every finding in this
report is `atemporal`, because the soybean table declares no time
or order axis. The other two values occur on dated data; the
[temporal analysis chapter](../methodology/temporal_analysis.ipynb)
works with them.

### Field: `quality_dimension`

The data-quality dimension, present only on data-quality findings.
Values: `completeness` (expected values are absent), `uniqueness`
(values or rows repeat that should identify), and `validity`
(values are recorded in inconsistent or implausible forms). In
this report the validity findings are format statements and the
uniqueness finding is a duplicate-row statement. No finding here
carries `completeness`.

In [5]:
example("quality_dimension", "validity")
example("quality_dimension", "uniqueness")

quality_dimension = validity: 10 findings
  [formats/format_variation] column area_damaged mixes value formats (3: 49% 'aaaaa-aaaaa' vs 33% 'aaa-aaaaa')
quality_dimension = uniqueness: 1 findings
  [duplicates/duplicate_rows] 52 rows repeat earlier values across the compared columns (41 groups, largest 4)


### Field: `status`

Whether the finding reports a structure that holds, or a departure
from an expectation. Values: `regularity` and `deviation`. The
soybean report holds both — dependencies and concentrations are
regularities; format mixes, duplicate rows, and values that break
their context's pattern are deviations.

In [6]:
example("status", "regularity")
example("status", "deviation")

status = regularity: 70 findings
  [dependencies/functional] int_discolor → sclerotia (sclerotia is a function of int_discolor; 0 violations in 645 rows)
status = deviation: 30 findings
  [intervals/categorical_outlier] within leafspots_marg = w-s-marg and canker_lesion = dna and external_decay = absent (233 rows), stem is norm in 99.6% of rows — 1 row breaks the pattern: row 272 (stem = abnorm)


## One pair, one differing field

The clearest view of what the fields add is a pair of findings that
the old scheme filed in different categories. The cell below finds
an exact functional dependency and an exact conditional rule in
this report whose facets differ in exactly one field, and prints
the two facet records side by side.

In [7]:
def facet_pair():
    fds = [f for f in report.findings
           if f["type"] == "functional" and f["facets"]["evidence"] == "exact"]
    rules = [f for f in report.findings
             if f["type"] == "if_then" and f["facets"]["evidence"] == "exact"]
    for a in fds:
        for b in rules:
            differing = [k for k in a["facets"]
                         if a["facets"][k] != b["facets"].get(k)]
            if differing == ["conditionality"]:
                return a, b
    raise LookupError("no pair differing only in conditionality")

fd, rule = facet_pair()
print("functional dependency:", fd["claim"])
print("conditional rule:     ", rule["claim"])
print()
print(pl.DataFrame({
    "field": list(fd["facets"]),
    "dependency": [str(fd["facets"][k]) for k in fd["facets"]],
    "rule": [str(rule["facets"][k]) for k in fd["facets"]],
}))

functional dependency: int_discolor → sclerotia (sclerotia is a function of int_discolor; 0 violations in 645 rows)
conditional rule:      When fruit_pods = "norm" and stem = "norm", fruit_spots = "absent" holds in 60 of 60 matching diagnostic rows (100.00%; baseline 59.56%)

┌────────────────┬───────────────┬───────────────────┐
│ field          ┆ dependency    ┆ rule              │
╞════════════════╪═══════════════╪═══════════════════╡
│ scope          ┆ columns       ┆ columns           │
│ object         ┆ relation      ┆ relation          │
│ evidence       ┆ exact         ┆ exact             │
│ conditionality ┆ unconditional ┆ value_conditioned │
│ temporality    ┆ atemporal     ┆ atemporal         │
│ status         ┆ regularity    ┆ regularity        │
└────────────────┴───────────────┴───────────────────┘


Read down the two columns: the statements have the same scope, the
same object, the same evidence standard, the same temporality, and
the same status. The dependency states that knowing `int_discolor`
settles `sclerotia` on every evaluated row. The rule states a
mapping of the same shape that holds under stated conditions on
`fruit_pods` and `stem`. One field, `conditionality`, records the
entire difference — and, through the derivation below, that one
field is also the entire reason the two findings carry different
categories.

## How the category is derived

The derivation is a fixed table of 18 rules. The rules are tried
in order against a finding's facets, and the first rule that
matches gives the category. One input is not a facet: whether the
relation is *directed* — the finding's subject names a target, or
its specification states an outcome. Quality rules come first, so
a data-quality statement is filed by its dimension before its kind
is considered.

| Rule | Condition on the facets | Category |
|---|---|---|
| 1 | `quality_dimension` is completeness, and `object` is distribution | `missingness` |
| 2 | `quality_dimension` is completeness, and `object` is anything else | `gaps` |
| 3 | `quality_dimension` is uniqueness, and `status` is regularity | `uniqueness` |
| 4 | `quality_dimension` is uniqueness, and `status` is deviation | `duplicates` |
| 5 | `quality_dimension` is validity | `formats` |
| 6 | `object` is role | `column_roles` |
| 7 | `object` is relation, and `conditionality` is value_conditioned | `conditional_rules` |
| 8 | `temporality` is change_across_segments, and `object` is not relation | `changes` |
| 9 | `object` is relation, and `status` is deviation | `changes` |
| 10 | `object` is relation, and the relation is directed | `dependencies` |
| 11 | `object` is relation, not directed, and `evidence` is exact | `column_roles` |
| 12 | `object` is relation, not directed, and `evidence` is not exact | `associations` |
| 13 | `object` is temporal_behaviour | `changes` |
| 14 | `object` is distribution, and `temporality` is ordered | `intervals` |
| 15 | `object` is distribution, and `status` is deviation | `intervals` |
| 16 | `object` is distribution, and `status` is regularity | `concentration` |
| 17 | `object` is structure, and `temporality` is ordered | `ordering` |
| 18 | `object` is structure | `gaps` |

Rule 8 excludes relations on purpose: a period association, whose
strength changes across periods, stays an association under rule
12 rather than becoming a change. Rule 9 exists for the step
discontinuity — a relation reported as a departure, where the level
on one side of a boundary does not continue the other side.

Walk the pair through the table. The dependency has no
`quality_dimension`, so rules 1–5 do not apply. Its `object` is
relation rather than role (rule 6 does not apply), it is
unconditional (rule 7 does not apply), atemporal (rule 8), and a
regularity (rule 9). Rule 10 matches — a directed relation — and
gives `dependencies`. The rule takes the same path until rule 7,
where its `value_conditioned` facet matches first and gives
`conditional_rules`. The order of the table is the mechanism: the
condition is tested before directedness, so a conditioned mapping
is a rule even though it also has an outcome.

The table is enforced, not advisory. A finding whose stated
category differs from the derived one fails validation when the
report is built, and the test suite pins the derivation for every
finding pattern. The cell below restates the table in Python and
checks it against every finding in this report:

In [8]:
def derive_category(f):
    x = f["facets"]
    q = x.get("quality_dimension")
    directed = "target" in f["subject"] or "outcome" in f["specification"]
    if q == "completeness" and x["object"] == "distribution":
        return "missingness"
    if q == "completeness":
        return "gaps"
    if q == "uniqueness" and x["status"] == "regularity":
        return "uniqueness"
    if q == "uniqueness":
        return "duplicates"
    if q == "validity":
        return "formats"
    if x["object"] == "role":
        return "column_roles"
    if x["object"] == "relation" and x["conditionality"] == "value_conditioned":
        return "conditional_rules"
    if x["temporality"] == "change_across_segments" and x["object"] != "relation":
        return "changes"
    if x["object"] == "relation" and x["status"] == "deviation":
        return "changes"
    if x["object"] == "relation" and directed:
        return "dependencies"
    if x["object"] == "relation" and x["evidence"] == "exact":
        return "column_roles"
    if x["object"] == "relation":
        return "associations"
    if x["object"] == "temporal_behaviour":
        return "changes"
    if x["object"] == "distribution" and x["temporality"] == "ordered":
        return "intervals"
    if x["object"] == "distribution" and x["status"] == "deviation":
        return "intervals"
    if x["object"] == "distribution":
        return "concentration"
    if x["object"] == "structure" and x["temporality"] == "ordered":
        return "ordering"
    return "gaps"

checked = sum(derive_category(f) == f["category"] for f in report.findings)
print(f"{checked} of {len(report.findings)} findings carry the category "
      "the table derives from their facets")

100 of 100 findings carry the category the table derives from their facets


## What the fields add for a reader

Category and type filtering work exactly as they did in schema
0.2, so code written against the old categories needs no changes.
The facets add the cuts the category cannot express. `select()`
takes one keyword per facet, and `findings_frame()` carries the
facets as columns:

In [9]:
relations = report.select(object="relation")
print(f"{len(relations)} findings state a relation between columns, "
      "across every category")

deviations = report.select(status="deviation")
print(f"{len(deviations)} findings report a deviation")

conditioned = report.select(conditionality="value_conditioned")
print(f"{len(conditioned)} findings hold under a condition on values")

frame = report.findings_frame()
print(frame.select(["category", "type", "object", "conditionality", "claim"]).head(4))

62 findings state a relation between columns, across every category
30 findings report a deviation
35 findings hold under a condition on values


┌───────────────┬─────────────────────┬──────────────┬────────────────┬────────────────────────────┐
│ category      ┆ type                ┆ object       ┆ conditionality ┆ claim                      │
╞═══════════════╪═════════════════════╪══════════════╪════════════════╪════════════════════════════╡
│ dependencies  ┆ functional          ┆ relation     ┆ unconditional  ┆ int_discolor → sclerotia   │
│               ┆                     ┆              ┆                ┆ (scle…                     │
│ dependencies  ┆ functional          ┆ relation     ┆ unconditional  ┆ fruit_spots, class →       │
│               ┆                     ┆              ┆                ┆ fruit_pod…                 │
│ concentration ┆ value_concentration ┆ distribution ┆ unconditional  ┆ column mycelium is nearly  │
│               ┆                     ┆              ┆                ┆ cons…                      │
│ dependencies  ┆ functional          ┆ relation     ┆ unconditional  ┆ class, leafspots_ha

The second count is the practical difference. Deviations —
findings that say the data departs from an expectation — live in
several categories at once: format mixes under `formats`,
duplicate rows under `duplicates`, context-breaking values under
`intervals`. No category collects them; the `status` field does.

## Where the scheme is weak

### Weak point 1: rule 11 files an exact symmetric relation under `column_roles`

Rule 11 is the derivation's awkward cell, and this report contains
two instances of it. The `duplicate_encoding` findings below state
an exact relation between two columns: every level of the finer
column maps to exactly one level of the coarser one, on every row.
The finding's subject names no target — the statement is not that
one column determines the other, in the way an input determines an
output. Rules 10 and 12 therefore do not
apply, and rule 11 files the finding under `column_roles`, on the
reasoning that the statement is about what the columns are — two
encodings of one quantity — rather than about a mapping or an
association strength.

In [10]:
for f in report.findings:
    if f["type"] == "duplicate_encoding":
        print(f"[{f['category']}] {f['claim']}")
        print("   facets:", f["facets"])

[column_roles] class rolls up to int_discolor: each of its 16 levels maps to one of 3 levels, on all 645 rows
   facets: {'scope': 'columns', 'object': 'relation', 'evidence': 'exact', 'conditionality': 'unconditional', 'temporality': 'atemporal', 'status': 'regularity'}
[column_roles] class rolls up to leaf_mild: each of its 15 levels maps to one of 3 levels, on all 575 rows
   facets: {'scope': 'columns', 'object': 'relation', 'evidence': 'exact', 'conditionality': 'unconditional', 'temporality': 'atemporal', 'status': 'regularity'}


The reasoning is defensible, and the filing still reads oddly: a
statement about a relation between two columns sits in the
category a reader expects to hold statements about single columns'
roles. If a second exact, symmetric relation pattern appears, rule
11 is the first cell to revisit.

### Weak point 2: assignment is per pattern first, per finding second

Facets are assigned by finding pattern, and the finding's own
record refines them in only the four cases listed at the top of
this chapter. A finding whose record carries a property the
derivation does not read — for example, an association computed
within periods but reported as one number — keeps its pattern's
default facets. The fields describe the pattern faithfully; they
do not yet describe every individual finding completely.

### Weak point 3: the `inferred` boundary is the least sharp definition

`evidence` separates `exact`, `approximate`, and `model_based` by
tests a reader can apply: exceptions counted, a rate or tolerance
measured, a model fitted. `inferred` is separated from
`model_based` by judgment instead. A role suggestion is an
interpretation of counts; a predictive dependency rests on a
fitted model; a future finding that interprets a model's output
would sit between the two, and the scheme does not yet say which
side it belongs on. Only the column-role findings use `inferred`
today, so the boundary has not yet been stressed.

## References

- [Reading findings](findings.ipynb) — the finding fields in
  report order, with the categories tabulated by type.
- [JSON report reference](../appendix/report_contract.ipynb) — the
  field-level contract for the report, including the `facets`
  object.
- [The findings reference](https://github.com/fcela/onceover/blob/main/docs/findings.md)
  — the facet assignment for every finding type.
- [The taxonomy design record](https://github.com/fcela/onceover/blob/main/docs/taxonomy-v1-design.md)
  — the engineering record behind this chapter, including the
  literature the scheme was checked against.